# Evaluación de MedSAM2 con pistas de hueso experto

**Objetivo.** Medir qué tan bien segmenta MedSAM2 (Ma et al., 2025) el sacro y los dos ilíacos en RM STIR de la articulación sacroilíaca cuando recibe como *pista* la máscara experta de unos pocos cortes (o una caja), comparando su salida con la máscara experta de los cortes que **no** recibió. El notebook corre el pipeline completo (`sij_medsam2`) para un conjunto de datos y muestra los resultados.

La sección 1 fija los parámetros; el resto se lee de arriba abajo.

## 1. Parámetros

**Qué se hace.** `DATASET` elige el conjunto: `"kaggle"` (15 pacientes, el análisis principal) o `"mendeley"` (3 pacientes, leídos de `data/raw/`). `DISPOSITIVO` es `"mps"` (GPU de Apple Silicon); si no hay MPS, el pipeline cae a CPU con un aviso, y en CPU el modelo es mucho más lento. `REHACER = False` reutiliza las predicciones ya guardadas en `data/processed/medsam2/<dataset>/`; con `True` se vuelve a correr el modelo.

In [ ]:
%matplotlib inline
import csv
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from sij_prep.io import repo_root
from sij_medsam2 import figuras
from sij_medsam2.datos import casos_seleccionados, leer_veredictos, stir_preprocesada
from sij_medsam2.modelo import elegir_dispositivo
from sij_medsam2.pipeline import dir_resultados, ejecutar_pipeline
from sij_medsam2.pistas import TIPOS_PISTA, construir_pistas, cortes_evaluacion, cortes_pista
from sij_medsam2.registro import mascara_experta

# --- Parámetros ---
DATASET = "kaggle"      # "kaggle" | "mendeley"
DISPOSITIVO = "mps"     # "mps" | "cpu"
REHACER = False         # True: vuelve a correr el modelo aunque haya predicciones guardadas

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    DISPOSITIVO_EFECTIVO = elegir_dispositivo(DISPOSITIVO).type
if DISPOSITIVO_EFECTIVO != DISPOSITIVO:
    print(f"AVISO: se pidió '{DISPOSITIVO}' pero se usará '{DISPOSITIVO_EFECTIVO}'. "
          "En CPU, correr el modelo es mucho más lento; las predicciones guardadas se reutilizan igual.")
DISPOSITIVO = DISPOSITIVO_EFECTIVO
SALIDA = dir_resultados(DATASET)
print(f"Dataset: {DATASET} | dispositivo: {DISPOSITIVO} | resultados en: {SALIDA.relative_to(repo_root())}")

## 2. Qué se mide

- **Entrada.** Cada volumen STIR (cortes de 3 mm, píxeles de 0.6 mm, 400×400) se normaliza y se pasa a MedSAM2. El corte 0 es el más anterior. Etiquetas de hueso: 1 = ilíaco, 2 = sacro, 3 = ilíaco contralateral.
- **Referencia experta.** Las máscaras de hueso de Kaggle (BoneLabels) fueron dibujadas en T1 y se llevaron a STIR por registro; solo se usan los cortes validados visualmente (el *núcleo* de cada caso, de 3 a 8 cortes).
- **Pistas.** MedSAM2 recibe la pista en uno o más cortes y propaga la segmentación hacia ambos lados sobre todo el volumen.
- **Evaluación.** Solo en los cortes del núcleo que **no** fueron pista. Si quedan menos de 2 cortes de evaluación, ese par (caso, tipo de pista) se excluye y se lista aparte. Ningún corte de pista entra en las métricas.
- **Métricas por hueso y corte.** Dice; HD95 (percentil 95 de la distancia de borde) y distancia media de borde, en px y en mm (1 px = 0.6 mm); si una de las dos máscaras está vacía, se cuenta el vacío y el HD95 queda indefinido.
- **Franja articular.** La región de la articulación se deriva de los huesos (sacro junto a cada ilíaco), limitada a las filas donde ambos huesos existen y están a 15 px o menos. Se informa su Dice y, como métrica principal de la articulación, la **distancia entre ejes de interlínea** (mediana y p95). `frac_filas_discordantes` es la fracción de filas donde predicción y experto no coinciden en tener franja.
- **Agregación.** Por caso y tipo de pista; luego por paciente (los casos 4 y 41 son el mismo paciente y se promedian juntos); luego media ± desviación estándar y mediana [p25–p75] entre pacientes.

## 3. Datos seleccionados

**Qué se hace.** Se listan los casos que entran en el conjunto elegido: los que tienen veredicto utilizable en la validación de la máscara experta. En `mendeley` quedan solo los que existen en Mendeley, sin repetir el paciente de los casos 4 y 41.

In [ ]:
casos = casos_seleccionados(DATASET, leer_veredictos())
filas = ["| Caso n | Paciente | Cortes del núcleo | N.º cortes |", "|---:|---|---|---:|"]
for c in casos:
    filas.append(f"| {c['n']} | {c['paciente']} | {' '.join(map(str, sorted(c['nucleo'])))} | {len(c['nucleo'])} |")
display(Markdown("\n".join(filas)))
print(f"{len(casos)} casos, {len({c['paciente'] for c in casos})} pacientes.")

## 4. Pistas

**Qué se hace.** Para cada caso hay cuatro tipos de pista, fijados antes de correr y sin ajustar con los resultados:

- `caja`: una caja por hueso (el rectángulo de la máscara experta más 5 px de margen) en el corte medio del núcleo; es el protocolo del artículo de MedSAM2.
- `k1`, `k2`, `k3`: la máscara experta de cada hueso en 1, 2 o 3 cortes repartidos por cuantiles internos del núcleo.

La celda dibuja, para un caso de ejemplo, los cortes de pista de cada tipo (caja en amarillo o máscaras de color: ilíaco rojo, sacro azul, ilíaco contralateral verde). La figura no se guarda.

In [ ]:
N_EJEMPLO = 27 if any(c["n"] == 27 for c in casos) else casos[0]["n"]
caso = next(c for c in casos if c["n"] == N_EJEMPLO)
nucleo = sorted(caso["nucleo"])
imgs = stir_preprocesada(N_EJEMPLO, DATASET)
experto = mascara_experta(N_EJEMPLO)
COLORES = {1: (1, 0, 0), 2: (0, 0.4, 1), 3: (0, 0.8, 0.2)}

pistas_por_tipo = {t: cortes_pista(nucleo, t) for t in TIPOS_PISTA}
ncol = max(len(v) for v in pistas_por_tipo.values())
fig, ejes = plt.subplots(len(TIPOS_PISTA), ncol, figsize=(3.4 * ncol, 3.4 * len(TIPOS_PISTA)), squeeze=False)
for i, tipo in enumerate(TIPOS_PISTA):
    cortes = pistas_por_tipo[tipo]
    pistas = construir_pistas(experto, cortes, tipo)
    for j in range(ncol):
        ax = ejes[i, j]
        ax.axis("off")
        if j >= len(cortes):
            continue
        c = cortes[j]
        ax.imshow(imgs[c], cmap="gray")
        for h, p in pistas.get(c, {}).items():
            if "mascara" in p:
                rgba = np.zeros(p["mascara"].shape + (4,))
                rgba[p["mascara"]] = (*COLORES[h], 0.45)
                ax.imshow(rgba)
            else:
                x0, y0, x1, y1 = p["caja"]
                ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="gold", lw=1.5))
        ax.set_title(f"{tipo}: corte {c}", fontsize=10)
fig.suptitle(f"Cortes de pista de cada tipo, caso n = {N_EJEMPLO} (núcleo: cortes {nucleo[0]}-{nucleo[-1]})")
fig.tight_layout()
plt.show()
for t in TIPOS_PISTA:
    ev = cortes_evaluacion(nucleo, pistas_por_tipo[t])
    print(f"{t}: pista en {pistas_por_tipo[t]}, evaluación en {ev}")

## 5. Ejecución

**Qué se hace.** `ejecutar_pipeline` hace todo el trabajo: lee la STIR, obtiene la máscara experta, construye las pistas, corre MedSAM2 (solo si no hay una predicción guardada), calcula las métricas y escribe los CSV y `configuracion.json` en `resultados/medsam2/<dataset>/`. Las métricas se recalculan siempre desde las predicciones. La primera vez descarga el checkpoint de MedSAM2 (y, para Kaggle, el conjunto de datos) automáticamente.

In [ ]:
cfg = ejecutar_pipeline(DATASET, DISPOSITIVO, rehacer=REHACER)
print()
print(f"Casos: {cfg['n_casos']} | corridas con métricas: {cfg['n_corridas_con_metricas']} | "
      f"excluidas: {cfg['n_excluidos']} | corridas nuevas del modelo: {cfg['n_corridas_modelo_nuevas']}")
print(f"Dispositivo de esta corrida: {cfg['dispositivo_corrida']} | predicciones calculadas en: {cfg['dispositivos_prediccion']}")
print(f"Tiempo de modelo acumulado: {cfg['segundos_modelo_total']} s | tiempo total de esta corrida: {cfg['tiempo_total_s']} s")

## 6. Resultados

### 6.1 Tablas

**Qué se hace.** Se leen los CSV con el módulo `csv` y se muestran como tablas. La primera resume, entre pacientes y sobre el subconjunto común de pacientes (`comun`), el Dice, el HD95 y el error del eje de la articulación (en mm) por tipo de pista, con media ± desv y mediana [p25–p75]; la segunda repite la tabla con todos los pacientes (`todos`, secundaria); la tercera lista los pares (caso, tipo) excluidos.

In [ ]:
def leer_csv(nombre):
    with open(SALIDA / nombre, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def tabla(columnas, filas):
    t = ["| " + " | ".join(columnas) + " |", "|" + "|".join("---" for _ in columnas) + "|"]
    t += ["| " + " | ".join(str(x) for x in f) + " |" for f in filas]
    display(Markdown("\n".join(t)))

def f3(x):
    return "n/d" if x in ("", "nan") else f"{float(x):.3f}"

resumen_csv = leer_csv("resumen.csv")
tipos = [t for t in TIPOS_PISTA if any(r["tipo"] == t for r in resumen_csv)]
METRICAS_TABLA = [("dice", "Dice"), ("hd95_mm", "HD95 mm"), ("eje_mediana_mm", "Error del eje, mediana mm")]

def tabla_resumen(conjunto, titulo):
    resumen = [r for r in resumen_csv if r["conjunto"] == conjunto]
    filas = []
    for est in ("iliaco", "sacro", "iliaco_contra", "franja_A", "franja_B"):
        for t in tipos:
            fila = [est, t, next((r["n_pacientes"] for r in resumen if r["tipo"] == t and r["estructura"] == est), "")]
            for m, _ in METRICAS_TABLA:
                r = next((r for r in resumen if r["tipo"] == t and r["estructura"] == est and r["metrica"] == m), None)
                if r is None or r["media"] in ("", "nan"):
                    fila.append("n/d")
                else:
                    fila.append(f"{f3(r['media'])} ± {f3(r['desv'])}; {f3(r['mediana'])} [{f3(r['p25'])}–{f3(r['p75'])}]")
            filas.append(fila)
    display(Markdown(titulo))
    tabla(["Estructura", "Pista", "Pacientes"] + [f"{n}: media ± desv; mediana [p25–p75]" for _, n in METRICAS_TABLA], filas)

tabla_resumen("comun", "**Resumen entre pacientes, comparación principal entre tipos de pista** (conjunto `comun`: "
              "los pacientes con métricas en los cuatro tipos, para que la composición de la muestra no cambie entre tipos; "
              "`n/d` = la métrica no aplica a esa estructura)")
tabla_resumen("todos", "**Resumen entre pacientes, conjunto `todos` (secundario)**: cada tipo usa todos sus pacientes, "
              "así que la muestra puede diferir entre tipos y las diferencias entre k mezclan el efecto de la pista con el de la muestra")

excl = leer_csv("excluidos.csv")
display(Markdown(f"**Pares excluidos** ({len(excl)})"))
if excl:
    tabla(["Caso n", "Paciente", "Pista", "Motivo", "Cortes de pista"],
          [[e["n"], e["paciente"], e["tipo"], e["motivo"], e["cortes_pista"]] for e in excl])

### 6.2 Figuras

**Qué se hace.** `figuras.resumen` regenera, sin correr el modelo, tres figuras: error por tipo de pista y error en función de la distancia (en cortes) a la pista más cercana (ambas con el subconjunto común de pacientes, como dice su título), que quedan en `resultados/medsam2/<dataset>/`, y `ejemplos.png` (cortes con experto frente a predicción), que contiene imágenes de RM y por eso se guarda en `data/processed/medsam2/<dataset>/figuras/`, sin versionar.

In [ ]:
for ruta in figuras.resumen(SALIDA):
    display(Markdown(f"`{ruta.relative_to(repo_root())}`"))
    display(Image(filename=str(ruta)))

## 7. Limitaciones

- **No hay corrección humana.** La "corrección" se reemplazó por la diferencia contra la máscara experta; no se midió esfuerzo ni tiempo de corrección.
- **La referencia experta viene de un registro T1 a STIR.** Las máscaras se dibujaron en T1 y se llevaron a STIR, así que el error de la referencia se mezcla con el del modelo; solo se usan cortes validados visualmente.
- **Muestra pequeña.** 15 pacientes (3 en Mendeley), con núcleos de 3 a 8 cortes. Los resultados son exploratorios.
- **k ≥ 2 interpola y k = 1 extrapola.** Con dos o más pistas MedSAM2 propaga entre cortes conocidos; con una sola, hacia afuera. Al comparar k hay que leerlo así.
- **Licencia de los datos de Kaggle: "Unknown".** Por eso no se versionan imágenes, máscaras ni predicciones; solo tablas y figuras de métricas.
- **MPS en float32 frente a bfloat16.** El artículo usa bfloat16 en GPU CUDA; aquí se corre en float32 en MPS y los resultados pueden diferir ligeramente.
- **Los cortes de pista no se evalúan.** Las métricas cubren solo los cortes del núcleo que no fueron pista, no el volumen completo.
- **El HD95 medio es optimista cuando la predicción queda vacía.** Si MedSAM2 no predice nada en un corte donde el experto sí tiene hueso, el Dice vale 0 pero el HD95 queda indefinido (NaN) y ese corte sale de la media del HD95. Pasa en 4 cortes en Kaggle y 2 en Mendeley; el HD95 medio no refleja esos fallos (el Dice y el conteo `n_vacio_pred` sí).
- **Reescalado bilineal dentro de MedSAM2.** La máscara de pista y los logits de salida se reescalan de forma bilineal (no por vecino más cercano) entre 400×400 y la resolución del modelo. El efecto es despreciable (Dice en el corte de pista ≥ 0.97), pero la pista no se copia bit a bit.
